<a href="https://colab.research.google.com/github/RohanT17/multimodal-cancer-classification/blob/main/multimodal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Late Fusion Model
==================
Combines:
  - Tabular branch:  HANCOCK clinical + pathological + blood JSONs (multi-task)
  - Image branch:    Pre-extracted WSI EfficientNetB0 embeddings from TCGA-HNSC
                     (subsite labels only; see wsi_embeddings.npy / wsi_embeddings_labels.npy)

NOTE on cohorts: HANCOCK and TCGA-HNSC are DIFFERENT patient cohorts, so we
cannot do patient-paired fusion. We evaluate population-level late fusion on
the SHARED task (primary tumor subsite, 4 classes). Each branch is trained
independently on its own cohort. We then late-fuse their per-class probability
distributions on a held-out test set drawn from each cohort.

Fusion target: primary tumor subsite (the only shared label).
Outputs are saved to Figures/.
"""

import json
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore", category=UserWarning)

start_time = time.perf_counter()

os.makedirs("Figures", exist_ok=True)

# ============================================================
# CONFIG
# ============================================================
CLINICAL_PATH = "StructuredData/clinical_data.json"
PATHOLOGICAL_PATH = "StructuredData/pathological_data.json"
BLOOD_PATH = "StructuredData/blood_data.json"
WSI_EMB_PATH = "wsi_embeddings.npy"
WSI_LBL_PATH = "wsi_embeddings_labels.npy"

RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2
BATCH_SIZE = 16
MAX_EPOCHS = 200
PATIENCE = 40

TARGETS = ["hpv_association_p16", "primary_tumor_site", "grading"]

LEAKY_COLS = [
    "pT_stage", "pN_stage", "histologic_type", "perinodal_invasion",
    "lymphovascular_invasion_L", "vascular_invasion_V",
    "perineural_invasion_Pn", "resection_status",
    "resection_status_carcinoma_in_situ", "carcinoma_in_situ",
]

LOSS_WEIGHTS = {"hpv_out": 1.0, "site_out": 1.5, "grade_out": 1.0}

# Image branch was trained on these 4 canonical subsites
SUBSITE_CANONICAL = ["Oral Cavity", "Larynx", "Oropharynx", "Hypopharynx"]
SUBSITE_TO_IDX = {s: i for i, s in enumerate(SUBSITE_CANONICAL)}

SEEDS = [0, 1, 2, 3, 4]


# ============================================================
# DATA LOADING — TABULAR
# ============================================================
def load_json_to_df(path: str) -> pd.DataFrame:
    with open(path) as f:
        return pd.json_normalize(json.load(f))


def load_structured_data() -> pd.DataFrame:
    """Load and merge HANCOCK clinical + pathological + blood data."""
    clinical = load_json_to_df(CLINICAL_PATH)
    pathological = load_json_to_df(PATHOLOGICAL_PATH)
    blood_raw = load_json_to_df(BLOOD_PATH)

    blood_raw.columns = blood_raw.columns.str.strip()
    clinical.columns = clinical.columns.str.strip()
    pathological.columns = pathological.columns.str.strip()

    blood = blood_raw.pivot_table(
        index="patient_id", columns="analyte_name",
        values="value", aggfunc="mean",
    ).reset_index()
    blood.columns.name = None

    df = (
        clinical
        .merge(pathological, on="patient_id", how="inner")
        .merge(blood, on="patient_id", how="inner")
    )
    print(f"Merged HANCOCK shape: {df.shape}")

    # Same data fix as text_model.py
    df["grading"] = df["grading"].replace("hpv_association_p16", "G_unknown")
    df = df.dropna(subset=TARGETS)
    print(f"Shape after dropna on targets: {df.shape}")
    return df


def map_site_to_subsite_idx(site_value):
    """Project HANCOCK primary_tumor_site value into one of the 4 canonical
    subsites used by the image branch. Returns -1 if unmappable.
    """
    s = str(site_value).strip().lower()
    if any(k in s for k in ["oral", "tongue", "lip", "gum",
                            "palate", "mouth", "cheek"]):
        return SUBSITE_TO_IDX["Oral Cavity"]
    if any(k in s for k in ["larynx", "glottis"]):
        return SUBSITE_TO_IDX["Larynx"]
    if any(k in s for k in ["oropharynx", "tonsil", "epiglottis"]):
        return SUBSITE_TO_IDX["Oropharynx"]
    if any(k in s for k in ["hypopharynx", "pyriform", "postcricoid"]):
        return SUBSITE_TO_IDX["Hypopharynx"]
    return -1


# ============================================================
# TABULAR PIPELINE
# ============================================================
def prep_tabular(df):
    """Patient-level split and feature engineering for the tabular branch."""
    X_all = df.drop(columns=TARGETS + ["patient_id"] + LEAKY_COLS, errors="ignore")
    y_all = df[TARGETS]

    pids = df["patient_id"].unique()
    train_pids, test_pids = train_test_split(
        pids, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    train_mask = df["patient_id"].isin(train_pids)
    test_mask = df["patient_id"].isin(test_pids)

    Xtr_raw, Xte_raw = X_all[train_mask].copy(), X_all[test_mask].copy()
    ytr_raw, yte_raw = y_all[train_mask].copy(), y_all[test_mask].copy()

    encoders, ytr_enc, yte_enc = {}, {}, {}
    for col in TARGETS:
        le = LabelEncoder().fit(y_all[col].astype(str))
        encoders[col] = le
        ytr_enc[col] = le.transform(ytr_raw[col].astype(str))
        yte_enc[col] = le.transform(yte_raw[col].astype(str))
        print(f"  {col}: {len(le.classes_)} classes")

    Xtr_oh = pd.get_dummies(Xtr_raw, dummy_na=False)
    Xte_oh = pd.get_dummies(Xte_raw, dummy_na=False)
    Xtr_oh, Xte_oh = Xtr_oh.align(Xte_oh, join="left", axis=1, fill_value=0)

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    Xtr = sc.fit_transform(imp.fit_transform(Xtr_oh)).astype(np.float32)
    Xte = sc.transform(imp.transform(Xte_oh)).astype(np.float32)

    sub_idx, val_idx = train_test_split(
        np.arange(len(Xtr)), test_size=VAL_SIZE,
        random_state=RANDOM_STATE,
        stratify=ytr_enc["hpv_association_p16"],
    )

    return {
        "X_tr": Xtr[sub_idx], "X_val": Xtr[val_idx], "X_te": Xte,
        "y_tr": {"hpv_out": ytr_enc["hpv_association_p16"][sub_idx],
                 "site_out": ytr_enc["primary_tumor_site"][sub_idx],
                 "grade_out": ytr_enc["grading"][sub_idx]},
        "y_val": {"hpv_out": ytr_enc["hpv_association_p16"][val_idx],
                  "site_out": ytr_enc["primary_tumor_site"][val_idx],
                  "grade_out": ytr_enc["grading"][val_idx]},
        "y_te": {"hpv_out": yte_enc["hpv_association_p16"],
                 "site_out": yte_enc["primary_tumor_site"],
                 "grade_out": yte_enc["grading"]},
        "encoders": encoders,
        "raw_test_site_values": yte_raw["primary_tumor_site"].values,
    }


def build_tabular_model(input_dim, encoders):
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(128, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    hpv = tf.keras.layers.Dense(
        len(encoders["hpv_association_p16"].classes_),
        activation="softmax", name="hpv_out")(x)
    site = tf.keras.layers.Dense(
        len(encoders["primary_tumor_site"].classes_),
        activation="softmax", name="site_out")(x)
    grade = tf.keras.layers.Dense(
        len(encoders["grading"].classes_),
        activation="softmax", name="grade_out")(x)

    model = tf.keras.Model(inputs=inputs, outputs=[hpv, site, grade])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss={k: "sparse_categorical_crossentropy"
              for k in ("hpv_out", "site_out", "grade_out")},
        loss_weights=LOSS_WEIGHTS,
        metrics={k: ["accuracy"]
                 for k in ("hpv_out", "site_out", "grade_out")},
    )
    return model


def get_sample_weights(y_tr):
    """Average per-task balanced sample weights (matches text_model.py)."""
    def cw(labels):
        cls = np.unique(labels)
        w = compute_class_weight("balanced", classes=cls, y=labels)
        return dict(zip(cls.tolist(), w.tolist()))

    hpv_cw, site_cw, grade_cw = cw(y_tr["hpv_out"]), cw(y_tr["site_out"]), cw(y_tr["grade_out"])
    return (
        np.array([hpv_cw[l] for l in y_tr["hpv_out"]])
        + np.array([site_cw[l] for l in y_tr["site_out"]])
        + np.array([grade_cw[l] for l in y_tr["grade_out"]])
    ) / 3.0


def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=PATIENCE,
            restore_best_weights=True, verbose=0, min_delta=1e-4),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=PATIENCE // 3, min_lr=1e-6, verbose=0),
    ]


# ============================================================
# IMAGE BRANCH (uses precomputed WSI embeddings)
# ============================================================
def load_wsi_data():
    if not (os.path.exists(WSI_EMB_PATH) and os.path.exists(WSI_LBL_PATH)):
        raise FileNotFoundError(
            f"Missing {WSI_EMB_PATH} or {WSI_LBL_PATH}. "
            "Run Tumor_Classification_Images.ipynb first to extract embeddings."
        )
    embs = np.load(WSI_EMB_PATH)
    labels = np.load(WSI_LBL_PATH)
    print(f"WSI embeddings: {embs.shape}, labels: {labels.shape}")
    print(f"  class distribution: {np.bincount(labels.astype(int))}")
    return embs.astype(np.float32), labels.astype(int)


def build_image_classifier(emb_dim, n_classes=4):
    inp = tf.keras.Input(shape=(emb_dim,))
    x = tf.keras.layers.Dense(64, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inp)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax",
                                name="subsite_out")(x)
    model = tf.keras.Model(inp, out, name="image_head")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def train_image_branch(X_emb, y_lbl, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    Xtr, Xte, ytr, yte = train_test_split(
        X_emb, y_lbl, test_size=TEST_SIZE,
        random_state=RANDOM_STATE, stratify=y_lbl,
    )
    Xtr_sub, Xval, ytr_sub, yval = train_test_split(
        Xtr, ytr, test_size=VAL_SIZE,
        random_state=RANDOM_STATE, stratify=ytr,
    )

    cls = np.unique(ytr_sub)
    w = compute_class_weight("balanced", classes=cls, y=ytr_sub)
    sw = np.array([dict(zip(cls.tolist(), w.tolist()))[l] for l in ytr_sub])

    model = build_image_classifier(X_emb.shape[1], n_classes=len(SUBSITE_CANONICAL))
    history = model.fit(
        Xtr_sub, ytr_sub,
        validation_data=(Xval, yval),
        sample_weight=sw,
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=make_callbacks(), verbose=0,
    )
    return model, Xte, yte, history


# ============================================================
# LATE FUSION
# ============================================================
def project_tabular_site_probs_to_subsite(site_probs, encoder):
    """Re-bin tabular per-class site probabilities into the 4 canonical subsites
    by grouping the HANCOCK site classes that fall into each bucket."""
    n_samples = site_probs.shape[0]
    out = np.zeros((n_samples, len(SUBSITE_CANONICAL)), dtype=np.float32)
    for cls_idx, cls_name in enumerate(encoder.classes_):
        bucket = map_site_to_subsite_idx(cls_name)
        if bucket >= 0:
            out[:, bucket] += site_probs[:, cls_idx]
    row_sum = out.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    return out / row_sum


def fuse_average(p_tab, p_img, w_tab=0.5, w_img=0.5):
    return w_tab * p_tab + w_img * p_img


def build_meta_classifier(input_dim, n_classes):
    inp = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(32, activation="relu")(inp)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    m = tf.keras.Model(inp, out)
    m.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m


# ============================================================
# EVALUATION
# ============================================================
def eval_subsite(name, probs, y_true, save_cm=True):
    y_pred = probs.argmax(axis=1)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    try:
        auc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")

    print(f"\n--- {name} ---")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Macro F1 : {f1:.4f}")
    print(f"  AUC-ROC  : {auc:.4f}")
    print(classification_report(y_true, y_pred,
        target_names=SUBSITE_CANONICAL, zero_division=0))

    if save_cm:
        cm = confusion_matrix(y_true, y_pred,
            labels=np.arange(len(SUBSITE_CANONICAL)))
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay(cm, display_labels=SUBSITE_CANONICAL).plot(
            ax=ax, cmap="Blues", values_format="d")
        ax.set_title(f"{name} Confusion Matrix")
        plt.xticks(rotation=30)
        safe = name.lower().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "")
        plt.savefig(f"Figures/late_{safe}_confusion.png", bbox_inches="tight")
        plt.close(fig)
    return {"acc": acc, "f1": f1, "auc": auc}


# ============================================================
# MAIN
# ============================================================
def main():
    # ====== TABULAR BRANCH ======
    print("\n" + "=" * 60 + "\nTABULAR BRANCH (HANCOCK)\n" + "=" * 60)
    df = load_structured_data()
    tab = prep_tabular(df)

    tab_results = []
    tab_test_site_probs_runs = []
    tab_history = None
    for seed in SEEDS:
        print(f"\n--- Tabular seed {seed} ---")
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)

        sw = get_sample_weights(tab["y_tr"])
        model = build_tabular_model(tab["X_tr"].shape[1], tab["encoders"])

        hist = model.fit(
            tab["X_tr"],
            [tab["y_tr"]["hpv_out"], tab["y_tr"]["site_out"],
             tab["y_tr"]["grade_out"]],
            validation_data=(tab["X_val"],
                [tab["y_val"]["hpv_out"], tab["y_val"]["site_out"],
                 tab["y_val"]["grade_out"]]),
            epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
            sample_weight=sw, callbacks=make_callbacks(), verbose=0,
        )
        tab_history = hist

        preds = model.predict(tab["X_te"], verbose=0)
        tab_test_site_probs_runs.append(preds[1])

        accs = [
            (preds[i].argmax(1) == tab["y_te"][k]).mean()
            for i, k in enumerate(["hpv_out", "site_out", "grade_out"])
        ]
        tab_results.append(accs)
        print(f"    seed {seed} -- HPV {accs[0]:.4f}  Site {accs[1]:.4f}  "
              f"Grade {accs[2]:.4f}")

    tab_results = np.array(tab_results)
    print(f"\nTabular task accuracies (mean ± std over {len(SEEDS)} seeds):")
    for i, t in enumerate(["HPV", "Site", "Grade"]):
        print(f"  {t}: {tab_results[:, i].mean():.4f} "
              f"± {tab_results[:, i].std():.4f}")

    tab_site_probs = np.mean(tab_test_site_probs_runs, axis=0)
    tab_subsite_probs = project_tabular_site_probs_to_subsite(
        tab_site_probs, tab["encoders"]["primary_tumor_site"])

    tab_test_subsite_true = np.array(
        [map_site_to_subsite_idx(s) for s in tab["raw_test_site_values"]])
    tab_keep = tab_test_subsite_true >= 0
    print(f"\nTabular test patients mappable to 4 canonical subsites: "
          f"{tab_keep.sum()}/{len(tab_keep)}")

    tab_subsite_probs_kept = tab_subsite_probs[tab_keep]
    tab_subsite_true_kept = tab_test_subsite_true[tab_keep]
    eval_subsite("Tabular only subsite", tab_subsite_probs_kept,
                 tab_subsite_true_kept)

    # ====== IMAGE BRANCH ======
    print("\n" + "=" * 60 + "\nIMAGE BRANCH (TCGA-HNSC WSI)\n" + "=" * 60)
    X_emb, y_lbl = load_wsi_data()

    img_results = []
    img_test_probs_runs = []
    img_test_y = None
    img_history = None
    for seed in SEEDS:
        print(f"  Image seed {seed}...")
        model, Xte, yte, hist = train_image_branch(X_emb, y_lbl, seed)
        img_history = hist
        probs = model.predict(Xte, verbose=0)
        img_test_probs_runs.append(probs)
        img_test_y = yte
        acc = (probs.argmax(1) == yte).mean()
        img_results.append(acc)
        print(f"    seed {seed} -- Acc {acc:.4f}")

    img_test_probs = np.mean(img_test_probs_runs, axis=0)
    print(f"\nImage subsite accuracy: "
          f"{np.mean(img_results):.4f} ± {np.std(img_results):.4f}")
    eval_subsite("Image only subsite", img_test_probs, img_test_y)

    # ====== LATE FUSION ======
    print("\n" + "=" * 60 + "\nLATE FUSION (population-level)\n" + "=" * 60)

    img_class_prior = img_test_probs.mean(axis=0, keepdims=True)
    img_prior_for_tab = np.tile(
        img_class_prior, (tab_subsite_probs_kept.shape[0], 1))

    fused_avg_tab = fuse_average(
        tab_subsite_probs_kept, img_prior_for_tab, w_tab=0.5, w_img=0.5)
    eval_subsite("Late fusion avg HANCOCK test", fused_avg_tab,
                 tab_subsite_true_kept)

    tab_class_prior = tab_subsite_probs_kept.mean(axis=0, keepdims=True)
    tab_prior_for_img = np.tile(
        tab_class_prior, (img_test_probs.shape[0], 1))

    fused_avg_img = fuse_average(
        tab_prior_for_img, img_test_probs, w_tab=0.5, w_img=0.5)
    eval_subsite("Late fusion avg TCGA test", fused_avg_img, img_test_y)

    # ====== META-CLASSIFIER FUSION ======
    print("\nTraining meta-classifier on stacked probabilities...")
    meta_X = np.concatenate([tab_prior_for_img, img_test_probs], axis=1)
    meta_y = img_test_y

    Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
        meta_X, meta_y, test_size=0.5,
        random_state=RANDOM_STATE, stratify=meta_y)

    meta = build_meta_classifier(meta_X.shape[1], n_classes=len(SUBSITE_CANONICAL))
    meta.fit(Xm_tr, ym_tr, validation_split=0.2,
             epochs=100, batch_size=BATCH_SIZE,
             callbacks=[tf.keras.callbacks.EarlyStopping(
                 patience=20, restore_best_weights=True, verbose=0)],
             verbose=0)
    meta_probs = meta.predict(Xm_te, verbose=0)
    eval_subsite("Late fusion meta TCGA test", meta_probs, ym_te)

    # ====== PLOTS ======
    if tab_history is not None:
        plt.figure(figsize=(8, 5))
        plt.plot(tab_history.history["loss"], label="Train")
        plt.plot(tab_history.history["val_loss"], label="Val")
        plt.xlabel("Epoch"); plt.ylabel("Loss")
        plt.title("Tabular Branch — Loss (final seed)")
        plt.legend(); plt.grid(True)
        plt.savefig("Figures/late_tabular_loss.png", bbox_inches="tight")
        plt.close()

    if img_history is not None:
        plt.figure(figsize=(8, 5))
        plt.plot(img_history.history["loss"], label="Train")
        plt.plot(img_history.history["val_loss"], label="Val")
        plt.xlabel("Epoch"); plt.ylabel("Loss")
        plt.title("Image Branch — Loss (final seed)")
        plt.legend(); plt.grid(True)
        plt.savefig("Figures/late_image_loss.png", bbox_inches="tight")
        plt.close()

    # ====== BASELINE ======
    print("\n" + "=" * 60 + "\nMAJORITY-CLASS BASELINE\n" + "=" * 60)
    dummy = DummyClassifier(strategy="most_frequent").fit(
        np.zeros((len(tab_subsite_true_kept), 1)), tab_subsite_true_kept)
    dummy_acc = dummy.score(
        np.zeros((len(tab_subsite_true_kept), 1)), tab_subsite_true_kept)
    print(f"Subsite majority-class accuracy (HANCOCK): {dummy_acc:.4f}")

    print(f"\nTOTAL RUNTIME: {time.perf_counter() - start_time:.2f} seconds")


if __name__ == "__main__":
    main()

In [ ]:
"""
Attention-Based Fusion Model
=============================
Fuses tabular (HANCOCK) and image (TCGA-HNSC WSI) probability outputs at the
embedding level using a learned per-sample attention layer. Trains the
attention + classifier head end-to-end on stacked modality embeddings.

The ModalityAttention layer learns per-sample weights over the two modalities
and exposes them at inference time for interpretability — i.e. for each test
patient we can see whether the model relied more on the tabular or image
modality.

Same cohort caveat as late_fusion.py: we evaluate on the TCGA test set with
the tabular branch contributing as a population-level prior (no patient
pairing exists between HANCOCK and TCGA-HNSC).

Outputs are saved to Figures/.
"""

import json
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore", category=UserWarning)

start_time = time.perf_counter()
os.makedirs("Figures", exist_ok=True)

# ============================================================
# CONFIG
# ============================================================
CLINICAL_PATH = "StructuredData/clinical_data.json"
PATHOLOGICAL_PATH = "StructuredData/pathological_data.json"
BLOOD_PATH = "StructuredData/blood_data.json"
WSI_EMB_PATH = "wsi_embeddings.npy"
WSI_LBL_PATH = "wsi_embeddings_labels.npy"

RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2
BATCH_SIZE = 16
MAX_EPOCHS = 200
PATIENCE = 40
EMBED_DIM = 32   # shared modality embedding dimension

TARGETS = ["hpv_association_p16", "primary_tumor_site", "grading"]
LEAKY_COLS = [
    "pT_stage", "pN_stage", "histologic_type", "perinodal_invasion",
    "lymphovascular_invasion_L", "vascular_invasion_V",
    "perineural_invasion_Pn", "resection_status",
    "resection_status_carcinoma_in_situ", "carcinoma_in_situ",
]
LOSS_WEIGHTS = {"hpv_out": 1.0, "site_out": 1.5, "grade_out": 1.0}

SUBSITE_CANONICAL = ["Oral Cavity", "Larynx", "Oropharynx", "Hypopharynx"]
SUBSITE_TO_IDX = {s: i for i, s in enumerate(SUBSITE_CANONICAL)}
SEEDS = [0, 1, 2, 3, 4]


# ============================================================
# DATA LOADING (shared with late_fusion.py)
# ============================================================
def load_json_to_df(path: str) -> pd.DataFrame:
    with open(path) as f:
        return pd.json_normalize(json.load(f))


def load_structured_data():
    clinical = load_json_to_df(CLINICAL_PATH)
    pathological = load_json_to_df(PATHOLOGICAL_PATH)
    blood_raw = load_json_to_df(BLOOD_PATH)

    blood_raw.columns = blood_raw.columns.str.strip()
    clinical.columns = clinical.columns.str.strip()
    pathological.columns = pathological.columns.str.strip()

    blood = blood_raw.pivot_table(
        index="patient_id", columns="analyte_name",
        values="value", aggfunc="mean").reset_index()
    blood.columns.name = None

    df = (
        clinical.merge(pathological, on="patient_id", how="inner")
                .merge(blood, on="patient_id", how="inner")
    )
    print(f"Merged HANCOCK shape: {df.shape}")
    df["grading"] = df["grading"].replace("hpv_association_p16", "G_unknown")
    df = df.dropna(subset=TARGETS)
    print(f"Shape after dropna on targets: {df.shape}")
    return df


def map_site_to_subsite_idx(site_value):
    s = str(site_value).strip().lower()
    if any(k in s for k in ["oral", "tongue", "lip", "gum",
                            "palate", "mouth", "cheek"]):
        return SUBSITE_TO_IDX["Oral Cavity"]
    if any(k in s for k in ["larynx", "glottis"]):
        return SUBSITE_TO_IDX["Larynx"]
    if any(k in s for k in ["oropharynx", "tonsil", "epiglottis"]):
        return SUBSITE_TO_IDX["Oropharynx"]
    if any(k in s for k in ["hypopharynx", "pyriform", "postcricoid"]):
        return SUBSITE_TO_IDX["Hypopharynx"]
    return -1


def prep_tabular(df):
    X_all = df.drop(columns=TARGETS + ["patient_id"] + LEAKY_COLS, errors="ignore")
    y_all = df[TARGETS]

    pids = df["patient_id"].unique()
    train_pids, test_pids = train_test_split(
        pids, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_mask = df["patient_id"].isin(train_pids)
    test_mask = df["patient_id"].isin(test_pids)

    Xtr_raw, Xte_raw = X_all[train_mask].copy(), X_all[test_mask].copy()
    ytr_raw, yte_raw = y_all[train_mask].copy(), y_all[test_mask].copy()

    encoders, ytr_enc, yte_enc = {}, {}, {}
    for col in TARGETS:
        le = LabelEncoder().fit(y_all[col].astype(str))
        encoders[col] = le
        ytr_enc[col] = le.transform(ytr_raw[col].astype(str))
        yte_enc[col] = le.transform(yte_raw[col].astype(str))

    Xtr_oh = pd.get_dummies(Xtr_raw, dummy_na=False)
    Xte_oh = pd.get_dummies(Xte_raw, dummy_na=False)
    Xtr_oh, Xte_oh = Xtr_oh.align(Xte_oh, join="left", axis=1, fill_value=0)

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    Xtr = sc.fit_transform(imp.fit_transform(Xtr_oh)).astype(np.float32)
    Xte = sc.transform(imp.transform(Xte_oh)).astype(np.float32)

    sub_idx, val_idx = train_test_split(
        np.arange(len(Xtr)), test_size=VAL_SIZE,
        random_state=RANDOM_STATE,
        stratify=ytr_enc["hpv_association_p16"])

    return {
        "X_tr": Xtr[sub_idx], "X_val": Xtr[val_idx], "X_te": Xte,
        "y_tr": {"hpv_out": ytr_enc["hpv_association_p16"][sub_idx],
                 "site_out": ytr_enc["primary_tumor_site"][sub_idx],
                 "grade_out": ytr_enc["grading"][sub_idx]},
        "y_val": {"hpv_out": ytr_enc["hpv_association_p16"][val_idx],
                  "site_out": ytr_enc["primary_tumor_site"][val_idx],
                  "grade_out": ytr_enc["grading"][val_idx]},
        "y_te": {"hpv_out": yte_enc["hpv_association_p16"],
                 "site_out": yte_enc["primary_tumor_site"],
                 "grade_out": yte_enc["grading"]},
        "encoders": encoders,
        "raw_test_site_values": yte_raw["primary_tumor_site"].values,
    }


def get_sample_weights(y_tr):
    def cw(labels):
        cls = np.unique(labels)
        w = compute_class_weight("balanced", classes=cls, y=labels)
        return dict(zip(cls.tolist(), w.tolist()))
    h, s, g = cw(y_tr["hpv_out"]), cw(y_tr["site_out"]), cw(y_tr["grade_out"])
    return (
        np.array([h[l] for l in y_tr["hpv_out"]])
        + np.array([s[l] for l in y_tr["site_out"]])
        + np.array([g[l] for l in y_tr["grade_out"]])
    ) / 3.0


def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=PATIENCE,
            restore_best_weights=True, verbose=0, min_delta=1e-4),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=PATIENCE // 3, min_lr=1e-6, verbose=0),
    ]


# ============================================================
# UNIMODAL MODELS (used to produce features for the attention model)
# ============================================================
def build_tabular_model(input_dim, encoders):
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(128, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    hpv = tf.keras.layers.Dense(
        len(encoders["hpv_association_p16"].classes_),
        activation="softmax", name="hpv_out")(x)
    site = tf.keras.layers.Dense(
        len(encoders["primary_tumor_site"].classes_),
        activation="softmax", name="site_out")(x)
    grade = tf.keras.layers.Dense(
        len(encoders["grading"].classes_),
        activation="softmax", name="grade_out")(x)

    model = tf.keras.Model(inputs, [hpv, site, grade])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss={k: "sparse_categorical_crossentropy"
              for k in ("hpv_out", "site_out", "grade_out")},
        loss_weights=LOSS_WEIGHTS,
        metrics={k: ["accuracy"]
                 for k in ("hpv_out", "site_out", "grade_out")},
    )
    return model


def build_image_classifier(emb_dim, n_classes=4):
    inp = tf.keras.Input(shape=(emb_dim,))
    x = tf.keras.layers.Dense(64, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inp)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    m = tf.keras.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def project_tabular_site_probs_to_subsite(site_probs, encoder):
    n = site_probs.shape[0]
    out = np.zeros((n, len(SUBSITE_CANONICAL)), dtype=np.float32)
    for cls_idx, cls_name in enumerate(encoder.classes_):
        bucket = map_site_to_subsite_idx(cls_name)
        if bucket >= 0:
            out[:, bucket] += site_probs[:, cls_idx]
    s = out.sum(axis=1, keepdims=True)
    s[s == 0] = 1.0
    return out / s


# ============================================================
# ATTENTION FUSION MODEL
# ============================================================
class ModalityAttention(tf.keras.layers.Layer):
    """Bahdanau-style additive attention over modality embeddings.

    Input  shape: (batch, n_modalities, embed_dim)
    Output shape: (batch, embed_dim)  — fused embedding
    Weights shape (saved internally on each call): (batch, n_modalities)
    """

    def __init__(self, attention_dim=16, **kwargs):
        super().__init__(**kwargs)
        self.attention_dim = attention_dim

    def build(self, input_shape):
        embed_dim = input_shape[-1]
        self.W = self.add_weight(
            shape=(embed_dim, self.attention_dim),
            initializer="glorot_uniform", name="W")
        self.v = self.add_weight(
            shape=(self.attention_dim, 1),
            initializer="glorot_uniform", name="v")
        super().build(input_shape)

    def call(self, embeddings):
        scores = tf.tanh(tf.matmul(embeddings, self.W))   # (B, M, A)
        scores = tf.matmul(scores, self.v)                # (B, M, 1)
        weights = tf.nn.softmax(scores, axis=1)           # (B, M, 1)
        fused = tf.reduce_sum(weights * embeddings, axis=1)
        return fused, tf.squeeze(weights, axis=-1)


def build_attention_fusion_model(tab_dim, img_dim, n_classes,
                                 embed_dim=EMBED_DIM):
    """Build two parallel models that share weights:
       - `train_model`:   returns only the classifier output (for fitting)
       - `inspect_model`: returns (probs, attention_weights) for analysis
    They share weights because both are constructed from the same layer
    instances; training train_model also trains inspect_model.
    """
    tab_inp = tf.keras.Input(shape=(tab_dim,), name="tab_input")
    img_inp = tf.keras.Input(shape=(img_dim,), name="img_input")

    tab_dense = tf.keras.layers.Dense(embed_dim, activation="relu",
                                      name="tab_embed")
    img_dense = tf.keras.layers.Dense(embed_dim, activation="relu",
                                      name="img_embed")
    drop1 = tf.keras.layers.Dropout(0.3)
    drop2 = tf.keras.layers.Dropout(0.3)
    attn_layer = ModalityAttention(attention_dim=16,
                                   name="modality_attention")
    head_dense = tf.keras.layers.Dense(32, activation="relu", name="head")
    head_drop = tf.keras.layers.Dropout(0.3)
    out_dense = tf.keras.layers.Dense(n_classes, activation="softmax",
                                      name="subsite_out")

    tab_emb = drop1(tab_dense(tab_inp))
    img_emb = drop2(img_dense(img_inp))
    stacked = tf.keras.layers.Lambda(
        lambda t: tf.stack(t, axis=1),
        name="stack_modalities")([tab_emb, img_emb])
    fused, attn_weights = attn_layer(stacked)
    x = head_drop(head_dense(fused))
    out = out_dense(x)

    train_model = tf.keras.Model(
        inputs=[tab_inp, img_inp], outputs=out,
        name="attention_fusion_train")
    train_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    inspect_model = tf.keras.Model(
        inputs=[tab_inp, img_inp],
        outputs=[out, attn_weights],
        name="attention_fusion_inspect",
    )
    return train_model, inspect_model


# ============================================================
# EVALUATION
# ============================================================
def eval_subsite(name, probs, y_true, save_cm=True):
    y_pred = probs.argmax(axis=1)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    try:
        auc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")
    print(f"\n--- {name} ---")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Macro F1 : {f1:.4f}")
    print(f"  AUC-ROC  : {auc:.4f}")
    print(classification_report(y_true, y_pred,
        target_names=SUBSITE_CANONICAL, zero_division=0))

    if save_cm:
        cm = confusion_matrix(y_true, y_pred,
            labels=np.arange(len(SUBSITE_CANONICAL)))
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay(cm, display_labels=SUBSITE_CANONICAL).plot(
            ax=ax, cmap="Blues", values_format="d")
        ax.set_title(f"{name} Confusion Matrix")
        plt.xticks(rotation=30)
        safe = name.lower().replace(" ", "_").replace("-", "_")
        plt.savefig(f"Figures/attn_{safe}_confusion.png", bbox_inches="tight")
        plt.close(fig)
    return {"acc": acc, "f1": f1, "auc": auc}


# ============================================================
# MAIN
# ============================================================
def main():
    # ====== Train tabular branch (HANCOCK) ======
    print("\n" + "=" * 60 + "\nTABULAR BRANCH (HANCOCK)\n" + "=" * 60)
    df = load_structured_data()
    tab = prep_tabular(df)

    tab_test_site_runs = []
    for seed in SEEDS:
        print(f"  Tabular seed {seed}...")
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)
        sw = get_sample_weights(tab["y_tr"])
        m = build_tabular_model(tab["X_tr"].shape[1], tab["encoders"])
        m.fit(
            tab["X_tr"],
            [tab["y_tr"]["hpv_out"], tab["y_tr"]["site_out"],
             tab["y_tr"]["grade_out"]],
            validation_data=(tab["X_val"],
                [tab["y_val"]["hpv_out"], tab["y_val"]["site_out"],
                 tab["y_val"]["grade_out"]]),
            epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
            sample_weight=sw, callbacks=make_callbacks(), verbose=0,
        )
        preds = m.predict(tab["X_te"], verbose=0)
        tab_test_site_runs.append(preds[1])

    tab_site_probs = np.mean(tab_test_site_runs, axis=0)
    tab_subsite_probs = project_tabular_site_probs_to_subsite(
        tab_site_probs, tab["encoders"]["primary_tumor_site"])

    # ====== Train image branch (TCGA) ======
    print("\n" + "=" * 60 + "\nIMAGE BRANCH (TCGA-HNSC)\n" + "=" * 60)
    if not (os.path.exists(WSI_EMB_PATH) and os.path.exists(WSI_LBL_PATH)):
        raise FileNotFoundError(
            f"Missing {WSI_EMB_PATH} or {WSI_LBL_PATH}. "
            "Run Tumor_Classification_Images.ipynb first to extract embeddings."
        )
    X_emb = np.load(WSI_EMB_PATH).astype(np.float32)
    y_lbl = np.load(WSI_LBL_PATH).astype(int)
    print(f"WSI embeddings: {X_emb.shape}, labels: {y_lbl.shape}")

    Xtr_img, Xte_img, ytr_img, yte_img = train_test_split(
        X_emb, y_lbl, test_size=TEST_SIZE,
        random_state=RANDOM_STATE, stratify=y_lbl)

    # ====== Build attention fusion training data ======
    # Tabular contributes a single class-prior vector (since cohorts differ),
    # broadcast to every TCGA test sample. Image contributes per-sample emb.
    tab_class_prior = tab_subsite_probs.mean(axis=0, keepdims=True)
    tab_for_img_train = np.tile(tab_class_prior, (Xtr_img.shape[0], 1))
    tab_for_img_test = np.tile(tab_class_prior, (Xte_img.shape[0], 1))

    # Hold out half of TCGA test for final fusion eval (the other half trains
    # the attention model since pure unimodal models already used Xtr_img).
    Xt_tr, Xt_te, yt_tr, yt_te, tabt_tr, tabt_te = train_test_split(
        Xtr_img, ytr_img, tab_for_img_train,
        test_size=0.3, random_state=RANDOM_STATE, stratify=ytr_img)

    # ====== Train attention fusion over 5 seeds ======
    print("\n" + "=" * 60 + "\nATTENTION FUSION\n" + "=" * 60)
    attn_results = []
    attn_test_probs_runs = []
    attn_test_weights_runs = []
    attn_history = None

    for seed in SEEDS:
        print(f"  Attention fusion seed {seed}...")
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)

        cls = np.unique(yt_tr)
        w = compute_class_weight("balanced", classes=cls, y=yt_tr)
        sw = np.array([dict(zip(cls.tolist(), w.tolist()))[l] for l in yt_tr])

        train_model, inspect_model = build_attention_fusion_model(
            tab_dim=tabt_tr.shape[1],
            img_dim=Xt_tr.shape[1],
            n_classes=len(SUBSITE_CANONICAL),
        )

        hist = train_model.fit(
            x=[tabt_tr, Xt_tr],
            y=yt_tr,
            validation_split=0.2,
            sample_weight=sw,
            epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
            callbacks=make_callbacks(), verbose=0,
        )
        attn_history = hist

        probs, weights = inspect_model.predict([tabt_te, Xt_te], verbose=0)
        attn_test_probs_runs.append(probs)
        attn_test_weights_runs.append(weights)
        acc = (probs.argmax(1) == yt_te).mean()
        attn_results.append(acc)
        print(f"    seed {seed} -- Acc {acc:.4f}  "
              f"mean attn (tab/img): "
              f"{weights.mean(0)[0]:.3f} / {weights.mean(0)[1]:.3f}")

    attn_probs = np.mean(attn_test_probs_runs, axis=0)
    attn_weights = np.mean(attn_test_weights_runs, axis=0)
    print(f"\nAttention fusion accuracy (TCGA held-out): "
          f"{np.mean(attn_results):.4f} ± {np.std(attn_results):.4f}")

    # ====== Evaluate ======
    eval_subsite("Attention fusion subsite", attn_probs, yt_te)

    # ====== Attention-weight analysis ======
    print("\n" + "=" * 60 + "\nATTENTION WEIGHT ANALYSIS\n" + "=" * 60)
    mean_w = attn_weights.mean(axis=0)
    print(f"  Average attention over test set: "
          f"tabular = {mean_w[0]:.3f}, image = {mean_w[1]:.3f}")

    # Per-class breakdown
    print("\n  Mean attention by true class:")
    for c, name in enumerate(SUBSITE_CANONICAL):
        mask = yt_te == c
        if mask.sum() == 0:
            continue
        cw = attn_weights[mask].mean(axis=0)
        print(f"    {name:13s} (n={mask.sum():3d}):  "
              f"tab {cw[0]:.3f}  img {cw[1]:.3f}")

    # Save attention-weight histogram
    plt.figure(figsize=(8, 4))
    plt.hist(attn_weights[:, 0], bins=20, alpha=0.7, label="Tabular")
    plt.hist(attn_weights[:, 1], bins=20, alpha=0.7, label="Image")
    plt.xlabel("Attention weight"); plt.ylabel("Number of test samples")
    plt.title("Per-sample attention weight distribution")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.savefig("Figures/attn_weight_histogram.png", bbox_inches="tight")
    plt.close()

    # Save attention loss curve
    if attn_history is not None:
        plt.figure(figsize=(8, 5))
        plt.plot(attn_history.history["loss"], label="Train")
        plt.plot(attn_history.history["val_loss"], label="Val")
        plt.xlabel("Epoch"); plt.ylabel("Loss")
        plt.title("Attention Fusion — Loss (final seed)")
        plt.legend(); plt.grid(True)
        plt.savefig("Figures/attn_loss.png", bbox_inches="tight")
        plt.close()

    # ====== Baseline ======
    dummy = DummyClassifier(strategy="most_frequent").fit(
        np.zeros((len(yt_te), 1)), yt_te)
    print(f"\nMajority-class baseline (TCGA test): "
          f"{dummy.score(np.zeros((len(yt_te), 1)), yt_te):.4f}")

    print(f"\nTOTAL RUNTIME: {time.perf_counter() - start_time:.2f} seconds")


if __name__ == "__main__":
    main()